In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
    "org.apache.iceberg:iceberg-aws-bundle:1.10.0,"
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0,"
    "org.postgresql:postgresql:42.7.3 "
    "pyspark-shell"
)

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
 
spark = (
    SparkSession.builder
    .appName("gold-showcase")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .config("spark.sql.catalog.postgres", "org.apache.spark.sql.execution.datasources.v2.jdbc.JDBCTableCatalog")
    .config("spark.sql.catalog.postgres.url", "jdbc:postgresql://postgres:5432/sourcedb")
    .config("spark.sql.catalog.postgres.driver", "org.postgresql.Driver")
    .config("spark.sql.catalog.postgres.user", os.environ.get("PG_USER", "cdc_user"))
    .config("spark.sql.catalog.postgres.password", os.environ.get("PG_PASSWORD", "cdc_pass"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

Spark 4.1.0


In [3]:
print(spark.table("lakehouse.cdc.bronze_customers").count())
print(spark.table("lakehouse.cdc.bronze_drivers").count())

from pyspark.sql import functions as F
spark.table("lakehouse.cdc.bronze_customers").select(
    F.countDistinct("after_id").alias("unique_after_id"),
    F.countDistinct("before_id").alias("unique_before_id"),
).show()


125
85
+---------------+----------------+
|unique_after_id|unique_before_id|
+---------------+----------------+
|             63|              25|
+---------------+----------------+



In [4]:
from pyspark.sql import functions as F, Window

bronze_df = spark.table("lakehouse.cdc.bronze_customers")

bronze_with_key = bronze_df.withColumn(
    "entity_id", F.coalesce(F.col("after_id"), F.col("before_id"))
)

w = Window.partitionBy("entity_id").orderBy(
    F.col("ts_ms").desc(), F.col("kafka_offset").desc()
)

deduped = (bronze_with_key
    .filter(F.col("op").isNotNull())
    .filter(F.col("entity_id").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

# Mitu on op='d' (kustutatud)?
deduped.groupBy("op").count().show()

+---+-----+
| op|count|
+---+-----+
|  d|   25|
|  c|   22|
|  u|   16|
+---+-----+



In [5]:
drivers = spark.read.table("lakehouse.cdc.silver_drivers").count()
zones = spark.read.table("lakehouse.taxi.gold_demand_patterns").select("pickup_zone").distinct().count()
avg_demand = spark.sql("SELECT AVG(avg_trip_count) as avg FROM lakehouse.taxi.gold_demand_patterns").collect()[0]["avg"]

print(f"Total drivers:       {drivers}")
print(f"Total zones:         {zones}")
print(f"Drivers per zone:    {drivers / zones:.4f}")
print(f"Avg demand per zone: {avg_demand:.4f}")

Total drivers:       29
Total zones:         136
Drivers per zone:    0.2132
Avg demand per zone: 43.4314


In [6]:
print(" Demand patterns overview ")
spark.sql("""
    SELECT
        pickup_zone,
        hour_of_day,
        ROUND(avg_trip_count, 2)    AS avg_trips,
        ROUND(stddev_trip_count, 2) AS std_dev,
        demand_classification
    FROM lakehouse.taxi.gold_demand_patterns
    ORDER BY avg_trips DESC
    LIMIT 10
""").show(truncate=False)

 Demand patterns overview 
+---------------------+-----------+---------+-------+---------------------+
|pickup_zone          |hour_of_day|avg_trips|std_dev|demand_classification|
+---------------------+-----------+---------+-------+---------------------+
|East Village         |2          |371.0    |0.0    |high demand          |
|East Village         |1          |330.0    |0.0    |high demand          |
|Lincoln Square East  |0          |306.0    |0.0    |high demand          |
|East Village         |0          |267.0    |0.0    |high demand          |
|Midtown Center       |0          |228.0    |0.0    |high demand          |
|Gramercy             |1          |226.0    |0.0    |high demand          |
|Upper East Side South|0          |225.0    |0.0    |high demand          |
|Clinton East         |2          |224.0    |0.0    |high demand          |
|Gramercy             |2          |214.0    |0.0    |high demand          |
|West Village         |0          |214.0    |0.0    |high dem

In [7]:
print("Distribution of demand classifications")
spark.sql("""
    SELECT
        demand_classification,
        COUNT(*) AS zone_hour_combinations
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY demand_classification
    ORDER BY zone_hour_combinations DESC
""").show()


Distribution of demand classifications
+---------------------+----------------------+
|demand_classification|zone_hour_combinations|
+---------------------+----------------------+
|               normal|                   339|
|          high demand|                    62|
+---------------------+----------------------+



In [8]:
print("Which 3 zones have the most predictable demand?")
spark.sql("""
    SELECT
        pickup_zone,
        ROUND(AVG(stddev_trip_count), 4) AS avg_std_dev,
        ROUND(AVG(avg_trip_count), 2)    AS avg_trips_per_hour,
        COUNT(*)                         AS hours_observed
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY pickup_zone
    HAVING COUNT(*) >= 3
    ORDER BY avg_std_dev ASC
    LIMIT 3
""").show(truncate=False)

Which 3 zones have the most predictable demand?
+---------------------+-----------+------------------+--------------+
|pickup_zone          |avg_std_dev|avg_trips_per_hour|hours_observed|
+---------------------+-----------+------------------+--------------+
|Upper East Side South|0.0        |131.0             |4             |
|Yorkville West       |0.0        |145.25            |4             |
|TriBeCa/Civic Center |0.0        |80.25             |4             |
+---------------------+-----------+------------------+--------------+



In [9]:
print("At what hour does demand peak city-wide?")
spark.sql("""
    SELECT
        hour_of_day,
        ROUND(SUM(avg_trip_count), 0)  AS total_avg_trips_citywide,
        COUNT(DISTINCT pickup_zone)    AS zones_active
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY hour_of_day
    ORDER BY total_avg_trips_citywide DESC
""").show(24, truncate=False)

At what hour does demand peak city-wide?
+-----------+------------------------+------------+
|hour_of_day|total_avg_trips_citywide|zones_active|
+-----------+------------------------+------------+
|0          |5545.0                  |94          |
|1          |5408.0                  |113         |
|2          |4486.0                  |101         |
|3          |1958.0                  |77          |
|23         |14.0                    |11          |
|20         |3.0                     |3           |
|4          |1.0                     |1           |
|21         |1.0                     |1           |
+-----------+------------------------+------------+



In [10]:
print("Supply-demand gap — underserved zones")
spark.sql("""
    SELECT
        pickup_zone,
        hour_of_day,
        ROUND(avg_trip_count, 2)     AS avg_demand,
        ROUND(drivers_available, 2)  AS drivers_available,
        ROUND(demand_supply_gap, 2)  AS gap,
        is_underserved
    FROM lakehouse.taxi.gold_supply_demand_gap
    WHERE is_underserved = TRUE
    ORDER BY demand_supply_gap DESC
    LIMIT 15
""").show(truncate=False)

Supply-demand gap — underserved zones
+---------------------+-----------+----------+-----------------+------+--------------+
|pickup_zone          |hour_of_day|avg_demand|drivers_available|gap   |is_underserved|
+---------------------+-----------+----------+-----------------+------+--------------+
|East Village         |2          |371.0     |2.4              |368.6 |true          |
|East Village         |1          |330.0     |1.77             |328.23|true          |
|Lincoln Square East  |0          |306.0     |1.6              |304.4 |true          |
|East Village         |0          |267.0     |1.4              |265.6 |true          |
|Midtown Center       |0          |228.0     |1.19             |226.81|true          |
|Gramercy             |1          |226.0     |1.21             |224.79|true          |
|Upper East Side South|0          |225.0     |1.18             |223.82|true          |
|Clinton East         |2          |224.0     |1.45             |222.55|true          |
|West

In [11]:
total        = spark.read.table("lakehouse.taxi.gold_demand_patterns").count()
underserved  = spark.sql("SELECT COUNT(*) as c FROM lakehouse.taxi.gold_supply_demand_gap WHERE is_underserved = TRUE").collect()[0]["c"]
total_gaps   = spark.read.table("lakehouse.taxi.gold_supply_demand_gap").count()
print(f"Total zone/hour combinations analysed : {total}")
print(f"Underserved zone/hour combinations    : {underserved} / {total_gaps}")
print(f"Underserved percentage                : {round(underserved/total_gaps*100, 1)}%")

Total zone/hour combinations analysed : 401
Underserved zone/hour combinations    : 385 / 401
Underserved percentage                : 96.0%


In [12]:
# Vaata mis op-id on Bronze-is mis EI ole Silver-is
spark.sql("""
    SELECT op, COUNT(*) as cnt
    FROM lakehouse.cdc.bronze_customers
    GROUP BY op
    ORDER BY op
""").show()

+---+---+
| op|cnt|
+---+---+
|  c| 48|
|  d| 25|
|  r| 15|
|  u| 37|
+---+---+



In [13]:
# 
spark.sql("SELECT COUNT(*) FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT COUNT(*) FROM lakehouse.cdc.bronze_customers").show()

+--------+
|count(1)|
+--------+
|      38|
+--------+

+--------+
|count(1)|
+--------+
|     125|
+--------+



In [14]:
# Is postgres and lakehouse in sync? If they are, then this returns an empty table
spark.sql("SELECT id FROM lakehouse.cdc.silver_drivers EXCEPT SELECT id FROM postgres.public.drivers;").show()

+---+
| id|
+---+
+---+



In [15]:
spark.stop()